In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

In [ ]:
from unsloth import FastModel
import torch

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",
    dtype = None, # None for auto detection
    max_seq_length = 4096, # Uzun RAG metinleri için 4096'ya çıkarıldı
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

In [ ]:
from transformers import TextStreamer
# Helper function for inference
def do_gemma_4_inference(messages, max_new_tokens = 128):
    _ = model.generate(
        **tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True, # Must add for generation
            tokenize = True,
            return_dict = True,
            return_tensors = "pt",
        ).to("cuda"),
        max_new_tokens = max_new_tokens,
        temperature = 1.0, top_p = 0.95, top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True),
        use_cache = True
    )

In [ ]:
sloth_link = "https://files.worldwildlife.org/wwfcmsprod/images/Sloth_Sitting_iStock_3_12_2014/story_full_width/8l7pbjmj29_iStock_000011145477Large_mini__1_.jpg"

messages = [{
    "role" : "user",
    "content": [
        { "type": "image", "image" : sloth_link },
        { "type": "text",  "text" : "Which films does this animal feature in?" }
    ]
}]
# You might have to wait 1 minute for Unsloth's auto compiler
do_gemma_4_inference(messages, max_new_tokens = 256)

In [ ]:
messages = [{
    "role": "user",
    "content": [{ "type" : "text",
                  "text" : "Write a poem about sloths." }]
}]
do_gemma_4_inference(messages)

In [ ]:
from IPython.display import Audio, display
Audio("https://www.nasa.gov/wp-content/uploads/2015/01/591240main_JFKmoonspeech.mp3")

In [ ]:
!wget -qqq https://www.nasa.gov/wp-content/uploads/2015/01/591240main_JFKmoonspeech.mp3 -O audio.mp3

In [ ]:
audio_file = "audio.mp3"

messages = [{
    "role" : "user",
    "content": [
        { "type": "audio", "audio" : audio_file },
        { "type": "text",  "text" : "What is this audio about?" }
    ]
}]
do_gemma_4_inference(messages, max_new_tokens = 256)

In [ ]:
messages = [{
    "role" : "user",
    "content": [
        { "type": "audio", "audio" : audio_file },
        { "type": "image", "image" : sloth_link },
        { "type": "text",  "text" : "What is this audio and image about? "\
                                    "How are they related?" }
    ]
}]
do_gemma_4_inference(messages, max_new_tokens = 256)

**finetune Gemma 4!**
You can finetune the vision and text parts for now through selection - the audio part can also be finetuned - we're working to make it selectable as well!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Data Prep

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("Metin/WikiRAG-TR", split = "train[:3000]")

In [ ]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

In [ ]:
dataset[100]

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for question, context, answer in zip(examples["question"], examples["context"], examples["answer"]):
        user_msg = f"Soru:\n{question}\n\nBağlam:\n{context}"

        convo = [
            {"role": "user", "content": user_msg},
            {"role": "model", "content": answer}
        ]

        formatted = tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        )

        texts.append(formatted)

    return {"text": texts}

In [ ]:
train_dataset[100]["text"]

In [ ]:
from trl import SFTTrainer, SFTConfig

# Veri setini SFTTrainer'a vermeden önce formatlayarak "text" sütununu oluşturuyoruz
train_dataset = dataset.map(formatting_prompts_func, batched = True, remove_columns = dataset.column_names)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset, # Formatlanmış veri setini kullanıyoruz
    eval_dataset = None,

    args = SFTConfig(
        dataset_text_field = "text",

        # 80GB VRAM avantajını kullanarak Batch Size ve Context Length artırıldı
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        max_seq_length = 4096,           # Sınırı 4096 yaptık, cevaplar artık kesilmeyecek
        packing = False,

        learning_rate = 2e-5,
        num_train_epochs = 2,
        warmup_steps = 50,

        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        weight_decay = 0.01,

        logging_steps = 10,
        seed = 3407,
        report_to = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# Modelin sadece asistanın (modelin) verdiği cevaplar üzerinden loss hesaplamasını sağlar
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

In [ ]:
# Eğitimin sadece cevaplar üzerinden yapıldığını doğrulamak için kontrol edelim.
# Soru kısmı modelin eğitimi sırasında kayıp (loss) hesaplamasından çıkarılmalıdır (-100 ile maskelenir).

print("--- TÜM GİRDİ (Input IDs) ---")
print(tokenizer.decode(trainer.train_dataset[100]["input_ids"]))

print("\n" + "="*50 + "\n")

print("--- MASKELENMİŞ HALİ (Labels - Sadece Cevap) ---")
# -100 olan (maskelenen) tokenları boşluk ile değiştirerek sadece modelin öğreneceği kısmı görüyoruz:
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " "))

# Let's train the model!

To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
trainer_stats = trainer.train()

### Modeli Test Etme (Inference)

In [ ]:
from unsloth.chat_templates import get_chat_template

# Chat şablonunu tekrar ayarlıyoruz (Gemma-4 için)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

# Çıkarım (Inference) hızını 2 kat artırmak için modeli optimize et
FastModel.for_inference(model)

# Eğitim formatımıza uygun bir test sorusu ve bağlamı hazırlıyoruz
test_question = "Yer Tengri kimin oğludur ve Gök Tengri ile ilişkisi nedir?"
test_context = "Tengri, Eski Türkçede Tanrı, Gökyüzü; Eski Türklerin ve Moğolların inancı Tengricilik'te Gök Tanrı (Kök Tengri) ya da Gök'ün yüce tinidir. Aynı zamanda Orhun Yazıtları'nda ilk çözümlenen sözcük olup yazılışı '𐱅𐰭𐰼𐰃' şeklindedir. Yer Tengri; Gök Tengri'nin torunu, Kayra Han'ın oğlu, Ülgen'in kardeşi ve Erlik'in amcası."

user_msg = f"Soru:\n{test_question}\n\nBağlam:\n{test_context}"

messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : user_msg,
    }]
}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Modelin cevap vermeye başlaması için gerekli
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

# Gemma-4 için önerilen ayarlarla cevap üretiyoruz
outputs = model.generate(
    **inputs,
    max_new_tokens = 256, # RAG cevapları için uzunluğu biraz artırdık
    temperature = 1.0,
    top_p = 0.95,
    top_k = 64,
)

# Sadece modelin ürettiği yeni cevabı (soru kısmı hariç) yazdır
print(" Modelin Cevabı:\n")
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

### Modeli Kaydetme (LoRA Adaptörleri)

In [ ]:
# 1. Eğitilen LoRA ağırlıklarını (sadece adaptörleri) Hugging Face'e hızlıca yükleme
print("LoRA adaptörleri Hugging Face'e yükleniyor...")
model.push_to_hub("uabali/gemma4-e4b-rag-TR", token = "YOUR_HF_TOKEN")
tokenizer.push_to_hub("uabali/gemma4-e4b-rag-TR", token = "YOUR_HF_TOKEN")
print("✅ LoRA adaptörleri başarıyla yüklendi! İşleminiz tamam.")

# ---------------------------------------------------------
# 2. VLLM için Float16 Formatında Birleştirme (VAKTİNİZ OLMADIĞI İÇİN KAPATILDI)
if False:
    model.save_pretrained_merged("gemma4-e4b-rag-TR-16bit", tokenizer)
    model.push_to_hub_merged("uabali/gemma4-e4b-rag-TR", tokenizer, token = "YOUR_HF_TOKEN")

# ---------------------------------------------------------
# 3. Llama.cpp / Ollama için GGUF Formatında Kaydetme (VAKTİNİZ OLMADIĞI İÇİN KAPATILDI)
if False:
    model.save_pretrained_gguf("gemma4-e4b-rag-TR-GGUF", tokenizer, quantization_method = "Q8_0")
    model.push_to_hub_gguf("uabali/gemma4-e4b-rag-TR-GGUF", tokenizer, quantization_method = "Q8_0", token = "YOUR_HF_TOKEN")

### 📥 Modelleri Google Drive'a Yedekleme (Yerel Cihaza İndirmek İçin)

In [ ]:
import os

# Google Drive'ınızda oluşturulacak klasörün adı
drive_hedef = "/content/drive/MyDrive/Gemma4_RAG_Modellerim"
os.makedirs(drive_hedef, exist_ok=True)

print("1. LoRA adaptörleri Drive'a kopyalanıyor...")
!cp -r /content/gemma4-e4b-rag-TR {drive_hedef}/

print("\n2. 16-bit Tam Model kopyalanıyor (Dosya boyutu büyük olduğu için bu işlem biraz sürebilir)...")
!cp -r /content/gemma4-e4b-rag-TR-16bit {drive_hedef}/

print("\n3. GGUF Formatlı Model kopyalanıyor...")
!cp -r /content/gemma4-e4b-rag-TR-GGUF* {drive_hedef}/

print(f"\n✅ Tüm işlemler tamamlandı! Modelleriniz Google Drive'ınızdaki '{drive_hedef}' klasörüne kopyalandı.")
print("Artık Google Drive'a girip bu dosyaları kendi bilgisayarınıza güvenle indirebilirsiniz.")

In [ ]:
%%capture
# Oturum sıfırlandığı için gerekli kütüphaneleri tekrar kuruyoruz
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
from unsloth import FastModel

print("0. Model ve Tokenizer belleğe geri yükleniyor (Daha önceki LoRA kaydınızdan)...")
model, tokenizer = FastModel.from_pretrained(
    model_name = "uabali/gemma4-e4b-rag-TR", # Dün HF'ye yüklediğiniz adaptör modeli
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

# Yeni Hugging Face repo adımız
repo_id = "uabali/gemma4-e4b-TR"
hf_token = "YOUR_HF_TOKEN" # Önceki hücrede kullandığınız token

print("\n1. 16-bit Tam Model (vLLM vb. için) birleştiriliyor ve Hugging Face'e yükleniyor...")
print("Bu işlem biraz vakit alabilir...")
model.push_to_hub_merged(
    repo_id,
    tokenizer,
    save_method="merged_16bit",
    token=hf_token
)

print("\n2. GGUF Formatlı Model (Ollama/Llama.cpp için) dönüştürülüp Hugging Face'e yükleniyor...")
model.push_to_hub_gguf(
    repo_id,
    tokenizer,
    quantization_method="q8_0",
    token=hf_token
)

print("\n✅ İşlem tamamlandı! Modelleriniz Hugging Face'e başarıyla yüklendi.")

### 🔄 Modeli Yükleme ve TextStreamer ile Çıkarım (Akış)

In [ ]:
# Kaydedilmiş LoRA adaptörlerini yükleyip test etmek isterseniz:
# (Model bellekte olmadığı için yerelden yükleme işlemini aktif ettik)
if True:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "gemma4-e4b-rag-TR", # Güncellediğimiz isim (yerel klasör)
        max_seq_length = 4096,           # RAG için belirlediğimiz bağlam uzunluğu
        load_in_4bit = True,
    )
    FastModel.for_inference(model) # Çıkarım hızlandırmasını aktifleştir

from transformers import TextStreamer

# TextStreamer ile modelin cevabını kelime kelime ekrana yazdıralım
test_question_2 = "Micrographia kimin eseridir ve önemi nedir?"
test_context_2 = "Micrographia Robert Hooke'un 1665 yılında yayımladığı eseridir. Mikroskopi hakkında yazılan ilk önemli eser olarak kabul edilir. Bu eser, Hooke'un yaptığı mikroskopik gözlemlere ve bu gözlemler ışığında yapılan böcek, bitki vb. çizimlere dayanır. Hücre sözcüğü ilk kez bu eserde kullanılmıştır."

user_msg_stream = f"Soru:\n{test_question_2}\n\nBağlam:\n{test_context_2}"

messages_stream = [{
    "role": "user",
    "content": [{"type" : "text", "text" : user_msg_stream}]
}]

inputs_stream = tokenizer.apply_chat_template(
    messages_stream,
    add_generation_prompt = True,
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

print("\n🤖 Modelin Cevabı (Canlı Akış):\n")
_ = model.generate(
    **inputs_stream,
    max_new_tokens = 256,
    temperature = 1.0,
    top_p = 0.95,
    top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)